In [27]:
import torch;
import torch.nn as nn;
import torch.optim as optim;
from torch.utils.data import Dataset, DataLoader;
import numpy as np;
import random;
import os;
import re;
import pickle;
from collections import Counter;

# 设置随机种子，保证实验可复现。
def set_seed(seed=42):
    torch.manual_seed(seed);
    torch.cuda.manual_seed_all(seed);
    np.random.seed(seed);
    random.seed(seed);
    torch.backends.cudnn.deterministic = True;
    torch.backends.cudnn.benchmark = False;

set_seed();

# 检查GPU可用性并设置设备。
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu');
print(f"使用设备: {device}");

if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}");
    print(f"GPU数量: {torch.cuda.device_count()}");
    print(f"当前GPU显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB");

使用设备: cuda
GPU型号: Tesla T4
GPU数量: 2
当前GPU显存: 14.56 GB


## 宏常量定义：

In [28]:
# 数据相关配置。
DATA_PATH = "/kaggle/input/datasets/ailananjuxi/xiyouji/xyj.txt";
# DATA_PATH = "./data/xyj.txt";
SEQ_LENGTH = 100; # 序列长度。
BATCH_SIZE = 1024; # 批次大小。

# 模型相关配置。
EMBED_DIM = 256;  # 嵌入维度。
HIDDEN_DIM = 512; # 隐藏层维度。
NUM_LAYERS = 3;   # LSTM层数。
DROPOUT = 0.3;    # Dropout概率。

# 训练相关配置。
EPOCHS = 10;
LEARNING_RATE = 0.001;
CLIP_GRAD = 5; # 梯度裁剪阈值。
PRINT_EVERY = 100;
SAVE_EVERY = 1; # 模型保存次数。

# 是否继续训练（从已有模型加载）。
CONTINUE_TRAIN = 1;
MODEL_SAVE_PATH = "./models/lstm_xiyouji.pth";
VOCAB_SAVE_PATH = "./models/vocab.pkl";

# 文本生成相关配置。
GENERATE_LENGTH = 500;
TEMPERATURE = 0.1;

# 创建模型保存目录。
os.makedirs("./models", exist_ok=True);

## 训练数据处理:

In [29]:
def load_and_preprocess_data(data_path): # 加载并预处理文本数据。
    
    # 读取文本文件。
    with open(data_path, 'r', encoding='utf-8') as f:
        text = f.read();
    
    # 清理文本：去除多余空白和特殊字符。
    text = re.sub(r'\s+', ' ', text);
    text = text.strip();
    
    print(f"文本总长度: {len(text)} 字符");
    print(f"唯一字符数: {len(set(text))}");
    print(f"\n文本前200字符:\n{text[:200]}");
    
    return text;

def save_vocab(char2idx, idx2char, save_path):# 保存词汇表到文件。
    vocab = {'char2idx': char2idx, 'idx2char': idx2char};
    with open(save_path, 'wb') as f:
        pickle.dump(vocab, f);
    print(f"词汇表已保存到: {save_path} ！");

def load_vocab(load_path): # 从文件加载词汇表。
    with open(load_path, 'rb') as f:
        vocab = pickle.load(f);
    print(f"词汇表已从 {load_path} 加载");
    return vocab['char2idx'], vocab['idx2char'];

text = load_and_preprocess_data(DATA_PATH);

文本总长度: 734950 字符
唯一字符数: 4198

文本前200字符:
﻿[(第1章 灵根育孕源流出 心性修持大道生)] 诗曰： 混沌未分天地乱，茫茫渺渺无人见。 自从盘古破鸿蒙，开辟从兹清浊辨。 覆载群生仰至仁，发明万物皆成善。 欲知造化会元功，须看《西游释厄传》。 盖闻天地之数，有十二万九千六百岁为一元。将一元分为十二会，乃子、丑、寅、卯、辰、巳、午、未、申、酉、戌、亥之十二支也。每会该一万八百岁。且就一日而论：子时得阳气而丑则鸡鸣，寅不通光而卯则日出，辰时食后而


In [30]:

class TextDataset(Dataset):
    def __init__(self, text, seq_length):
        self.seq_length = seq_length; # 序列长度。
        self.text = text; # 原始文本。
        
        # 构建字符到索引的映射。
        self.chars = sorted(list(set(text)));
        self.char2idx = {ch: i for i, ch in enumerate(self.chars)};
        self.idx2char = {i: ch for i, ch in enumerate(self.chars)};
        self.vocab_size = len(self.chars);
        
        # 将文本转换为索引序列。
        self.data = [self.char2idx[ch] for ch in text];
    
    def __len__(self):
        return len(self.data) - self.seq_length;
    
    def __getitem__(self, idx):
        # 获取输入序列和目标序列（目标序列是输入序列向后移动一位）。
        x = torch.tensor(self.data[idx:idx+self.seq_length], dtype=torch.long);
        y = torch.tensor(self.data[idx+1:idx+self.seq_length+1], dtype=torch.long);
        return x, y;

In [31]:
# 创建数据集。
dataset = TextDataset(text, SEQ_LENGTH);
vocab_size = dataset.vocab_size;
char2idx = dataset.char2idx;
idx2char = dataset.idx2char;

# 保存词汇表供后续使用。
save_vocab(char2idx, idx2char, VOCAB_SAVE_PATH);

# 创建数据加载器，使用多进程加速数据加载。
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
);

print(f"\n数据集大小: {len(dataset)} 样本");
print(f"词汇表大小: {vocab_size}");
print(f"批次数量: {len(dataloader)}");

词汇表已保存到: ./models/vocab.pkl ！

数据集大小: 734850 样本
词汇表大小: 4198
批次数量: 718


## 模型搭建：

In [32]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super(LSTMModel, self).__init__();
        
        self.hidden_dim = hidden_dim;
        self.num_layers = num_layers;
        self.vocab_size = vocab_size;
        
        # 词嵌入层，将字符索引转换为密集向量。
        self.embedding = nn.Embedding(vocab_size, embed_dim);
        
        # LSTM层，使用多层结构增强模型表达能力。
        self.lstm = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        );
        
        # Dropout层，防止过拟合。
        self.dropout = nn.Dropout(dropout);
        
        # 全连接层，将LSTM输出映射到词汇表空间。
        self.fc = nn.Linear(hidden_dim, vocab_size);
    
    def forward(self, x, hidden=None):
        # x形状: (batch_size, seq_length)
        batch_size = x.size(0);
        
        # 词嵌入。
        embed = self.embedding(x);  # (batch_size, seq_length, embed_dim)
        
        # LSTM前向传播。
        if hidden is None:
            lstm_out, hidden = self.lstm(embed);
        else:
            lstm_out, hidden = self.lstm(embed, hidden);
        # lstm_out形状: (batch_size, seq_length, hidden_dim)
        
        # Dropout。
        lstm_out = self.dropout(lstm_out);
        
        # 全连接层。
        output = self.fc(lstm_out);  # (batch_size, seq_length, vocab_size)
        
        return output, hidden;
    
    def init_hidden(self, batch_size):
        """初始化隐藏状态。"""
        weight = next(self.parameters());
        hidden = (
            weight.new_zeros(self.num_layers, batch_size, self.hidden_dim),
            weight.new_zeros(self.num_layers, batch_size, self.hidden_dim)
        );
        return hidden;

In [33]:
# 初始化模型。
model = LSTMModel(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT);
model = model.to(device);

# 使用DataParallel支持多GPU训练。
if torch.cuda.device_count() > 1:
    print(f"使用 {torch.cuda.device_count()} 块GPU进行训练！");
    model = nn.DataParallel(model);

# 打印模型结构。
print(model);

# 统计模型参数量。
total_params = sum(p.numel() for p in model.parameters());
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad);
print(f"\n总参数量: {total_params:,}");
print(f"可训练参数量: {trainable_params:,}");
print(f"模型大小: {total_params * 4 / 1024**2:.2f} MB");

使用 2 块GPU进行训练！
DataParallel(
  (module): LSTMModel(
    (embedding): Embedding(4198, 256)
    (lstm): LSTM(256, 512, num_layers=3, batch_first=True, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
    (fc): Linear(in_features=512, out_features=4198, bias=True)
  )
)

总参数量: 9,007,718
可训练参数量: 9,007,718
模型大小: 34.36 MB


In [34]:
def generate_text(model, start_text, length, device, temperature=0.1):

    model.eval(); # 切换到评估模式。

    # 处理多GPU模型，获取实际模型。
    if isinstance(model, nn.DataParallel):
        actual_model = model.module;
    else:
        actual_model = model;
    
    # 将起始文本转换为索引。
    input_seq = [char2idx.get(ch, 0) for ch in start_text];
    input_tensor = torch.tensor([input_seq], dtype=torch.long).to(device);
    
    generated = list(start_text);
    hidden = None;
    
    with torch.no_grad():
        # 预热模型，传入起始文本。
        if len(input_seq) > 0:
            output, hidden = model(input_tensor, hidden);
        
        # 逐个生成字符。
        for _ in range(length):
            # 取最后一个时间步的输出。
            logits = output[:, -1, :] / temperature;
            
            # 使用softmax获取概率分布。
            probs = torch.softmax(logits, dim=-1);
            
            # 采样下一个字符。
            next_char_idx = torch.multinomial(probs, 1).item();
            next_char = idx2char[next_char_idx];
            
            generated.append(next_char);
            
            # 准备下一个输入。
            input_tensor = torch.tensor([[next_char_idx]], dtype=torch.long).to(device);
            output, hidden = actual_model(input_tensor, hidden);

    model.train();
    return ''.join(generated);

## 训练：

In [35]:
def train_model(model, dataloader, epochs, learning_rate, device, continue_train=False):
    
    start_epoch = 0;
    
    # 如果是继续训练，加载已有模型。
    if continue_train and os.path.exists(MODEL_SAVE_PATH):
        print(f"从 {MODEL_SAVE_PATH} 加载模型继续训练...");
        checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device);
        
        # 处理DataParallel保存的模型。
        state_dict = checkpoint['model_state_dict'];
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(state_dict);
        else:
            model.load_state_dict(state_dict);
        
        start_epoch = checkpoint.get('epoch', 0);
        print(f"从第 {start_epoch} 轮继续训练");
    
    # 定义损失函数和优化器。
    criterion = nn.CrossEntropyLoss();
    optimizer = optim.Adam(model.parameters(), lr=learning_rate);
    
    # 如果是继续训练，加载优化器状态。
    if continue_train and os.path.exists(MODEL_SAVE_PATH):
        checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device);
        if 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict']);
    
    # 学习率调度器，当损失不再下降时降低学习率。
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    );
    
    # 混合精度训练，加速并减少显存占用。
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None;
    
    model.train();
    
    for epoch in range(start_epoch, epochs):
        total_loss = 0;
        batch_count = 0;
        
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            inputs = inputs.to(device, non_blocking=True);
            targets = targets.to(device, non_blocking=True);
            
            optimizer.zero_grad();
            
            # 使用混合精度训练。
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs, _ = model(inputs);
                    loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1));
                
                scaler.scale(loss).backward();
                
                # 梯度裁剪，防止梯度爆炸。
                scaler.unscale_(optimizer);
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD);
                
                scaler.step(optimizer);
                scaler.update();
            else:
                outputs, _ = model(inputs);
                loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1));
                loss.backward();
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD);
                optimizer.step();
            
            total_loss += loss.item();
            batch_count += 1;
            
            if (batch_idx + 1) % PRINT_EVERY == 0:
                avg_loss = total_loss / batch_count;
                print(f"Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(dataloader)}], Loss: {avg_loss:.4f}");
        
        # 计算本轮平均损失。
        epoch_loss = total_loss / batch_count;
        print(f"✅Epoch [{epoch+1}/{epochs}] 完成，平均损失: {epoch_loss:.4f}\n");
        
        # 更新学习率。
        scheduler.step(epoch_loss);
        
        # 定期保存模型。
        if (epoch + 1) % SAVE_EVERY == 0 or epoch == epochs - 1:
            save_checkpoint(model, optimizer, epoch + 1, epoch_loss);
            
            # 生成示例文本查看训练效果。
            print("\n生成示例文本:");
            sample = generate_text(model, "话说", 200, device, temperature=0.8);
            print(sample);
            print("-" * 50);
    
    return model;

In [36]:
def save_checkpoint(model, optimizer, epoch, loss):    
    # 获取实际模型（处理DataParallel包装的情况）。
    if isinstance(model, nn.DataParallel):
        model_state = model.module.state_dict();
    else:
        model_state = model.state_dict();
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    };
    
    torch.save(checkpoint, MODEL_SAVE_PATH);
    print(f"模型已保存到: {MODEL_SAVE_PATH}");

### 开始训练：

In [37]:
%%time

print("开始训练模型......");
print(f"继续训练模式: {CONTINUE_TRAIN}");
print("=" * 50);

trained_model = train_model(
    model, 
    dataloader, 
    EPOCHS, 
    LEARNING_RATE, 
    device,
    continue_train=CONTINUE_TRAIN
);

print("\n训练完成！");

开始训练模型......
继续训练模式: 1
从 ./models/lstm_xiyouji.pth 加载模型继续训练...
从第 6 轮继续训练


/tmp/ipykernel_55/3766778480.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None;
/tmp/ipykernel_55/3766778480.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [7/50], Batch [100/718], Loss: 2.6110
Epoch [7/50], Batch [200/718], Loss: 2.5977
Epoch [7/50], Batch [300/718], Loss: 2.5837
Epoch [7/50], Batch [400/718], Loss: 2.5699
Epoch [7/50], Batch [500/718], Loss: 2.5559
Epoch [7/50], Batch [600/718], Loss: 2.5425
Epoch [7/50], Batch [700/718], Loss: 2.5298
✅Epoch [7/50] 完成，平均损失: 2.5277

模型已保存到: ./models/lstm_xiyouji.pth

生成示例文本:
话说罢。”三藏道：“这怪知我这件，有斋错了，煮饭睡罢罢。” 行者笑道：“师父不要说嘴乱；再有些喜法、力号铁样儿。”行者道：“师父，我 也不是徒弟：我不是啊，你是个鼠儿不净，也不是个是世爷王，教我一般等尝， 要与你两处。你身之有?我可怜，诚日知，容醒之言，就把他们都去吃。”三藏 道：“不敢说。”八戒道：“既不归西，到西天拜，大开天府之寿。是我圣僧四众， 奉上大帝，奉求夫妻皇帝。为至贵劳，又为。”遂辞别恩
--------------------------------------------------
Epoch [8/50], Batch [100/718], Loss: 2.4343
Epoch [8/50], Batch [200/718], Loss: 2.4221
Epoch [8/50], Batch [300/718], Loss: 2.4112
Epoch [8/50], Batch [400/718], Loss: 2.3998
Epoch [8/50], Batch [500/718], Loss: 2.3892
Epoch [8/50], Batch [600/718], Loss: 2.3786
Epoch [8/50], Batch [700/718], Loss: 2.3682
✅Epoch [8/50] 完成，平均损失: 2.3663

模型已保存到: ./models/lstm_xiy

KeyboardInterrupt: 

## 测试:

In [40]:
def load_model_for_inference(model_path, vocab_path, device):    
    # 加载词汇表。
    global char2idx, idx2char;
    char2idx, idx2char = load_vocab(vocab_path);
    vocab_size = len(char2idx);
    
    # 创建模型。
    model = LSTMModel(vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT);
    model = model.to(device);
    
    # 加载模型权重。
    checkpoint = torch.load(model_path, map_location=device);
    state_dict = checkpoint['model_state_dict'];
    
    # 处理可能的DataParallel前缀。
    new_state_dict = {};
    for k, v in state_dict.items():
        name = k.replace('module.', '') if k.startswith('module.') else k;
        new_state_dict[name] = v;
    
    model.load_state_dict(new_state_dict);
    print(f"模型已从 {model_path} 加载");
    
    return model;

In [39]:
# 测试生成效果。
print("=" * 60);
print("LSTM文本生成测试");
print("=" * 60);

# 使用训练好的模型生成文本。
if os.path.exists(MODEL_SAVE_PATH):
    # 加载模型。
    inference_model = load_model_for_inference(MODEL_SAVE_PATH, VOCAB_SAVE_PATH, device);
    
    # 测试不同的起始文本和温度参数。
    test_prompts = ["话说", "悟空", "唐僧", "那妖怪"];
    temperatures = [0.5, 0.8, 1.0];
    
    for prompt in test_prompts:
        print(f"\n起始文本: '{prompt}'");
        print("-" * 60);
        
        for temp in temperatures:
            print(f"\n温度={temp}:");
            generated = generate_text(inference_model, prompt, GENERATE_LENGTH, device, temperature=temp);
            print(generated);
            print();
else:
    print(f"模型文件 {MODEL_SAVE_PATH} 不存在，请先训练模型！");

LSTM文本生成测试
词汇表已从 ./models/vocab.pkl 加载
模型已从 ./models/lstm_xiyouji.pth 加载

起始文本: '话说'
------------------------------------------------------------

温度=0.5:
话说，不知那里去，还 是甚么东土。”三藏道：“是东土大唐来的，因上宝华地灵宝，方知是那国主公之， 不知有何不故。”太宗道：“既是观音菩萨来与我做法师，就是那皇帝引着师父的礼 哩。”行者道：“那官来我不敢上，我的手边有三个六尺长，把那大唐围下做六个 金银与唐；却才在银驼山中下人，武灵里炼，曾被众神满天中，等我去寻他那几个 妖精，与他交战，未得胜负，今日你如何难敌！”那怪物闻言，心中大怒道：“泼魔 与我战，若肯与你赌斗，你就把我打倒在地，就就如何！”老魔道：“贤甚不可， 不若打我的，就如今与你手打的人，却好来捉！”唬得那三藏魂飞魄散，跌跌的来走了。 又叫几声：“猪八戒也也念风凶恶，只管把马，一齐跑来，师父宽心哭哩。”八戒道： “你怎么不认？”行者道：“你不知道。那个是那山高怪，妖精神通，特与我当 同求真之意，只教他一个言仇无量，早往山天彻地风，捉进那府门尽要降精。我 们只是天去落风，那山下无挂无雨，才不得他两家是师父的功绩。” 行者闻言，止不住泪如泉涌，双耳齐声，叫道：“泼猴儿!饶你罢!我在那里 不见半斗哩？”行者道：“师父说得有理!你看我的坐身，不曾跟你来见。”行者笑 道：“不是好事，不是这


温度=0.8:
话说!师父，不要动手。”那行者在底，坐不住 的，却就将钉的铁子高道：“我师徒且休乱欢，看看看。”行者听言，丢开手段，立在 旁上，叫道：“我的头哩!原来不曾见佛穿的衣，如何不曾看动?原在那山的身、衣 服，都是金银宝贝来，却就油油?一把血了衣裳，足上一段有二百余里。大圣却到 底下，分些毫眼不容，把公主一?吞香，喜乐于地毕，又走来报道：“主公，外面 东土和尚，在那里刺行的；因日年在风间里寻他将，在那里一件宝贝来也。”那罗刹 红倒在床上，眼巴里放下高张。行者暗暗笑道：“好好，妖精!拿了!若是请他们吃了， 唐僧也不干好了。这个老子来时还有，却请老孙的二个拿行哩。”八戒道：“要来问 甚么邪怪么？”八戒道：“你不曾看看头，先在那老里头儿。若是一个铁身流，也就 是死坏了；